# Police Force Crime Data Pipeline
## Phase 1 - Data Engineering

**Assignment:** Python Project - Crime Data Pipeline & Reporting Dataset  
**Reporting grain:** Police Force × Month × Crime Category  
**Forces covered:** Metropolitan Police Service, West Midlands Police, West Yorkshire Police, Devon & Cornwall Police  
**Time period:** April 2023 -> March 2026 (36 months)

---

This notebook takes raw UK Police street-level crime data, cleans and enriches it using three external datasets,combining it into a single aggregated CSV exported to the 'outputs/' folder.

## Section 0 - Setup & Configuration

Before working with the data, all the key settings are centralised in one place. 

Such as: 
- file paths
- force names
- population estimates
- keyword mappings (file detection)

Ensuring that the pipeline stays flexible and updatable.

In [4]:
# standard library
import os
import re
import warnings

# data manipulation 
import pandas as pd
import numpy as np

# utilities displayed
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("all imports loaded successfully!!")

all imports loaded successfully!!


In [5]:
# path configuration 
from pathlib import Path

# notebook sits at the project root (same level as crime-data/, extra-data/,
# and outputs/). Path(__file__) is not available in jupyter
# globals()['_dh'][0] which always resolves to the folder containing the .ipynb file no matter where Jupyter was launched from.
# ---------

BASE_DIR  = Path(globals()['_dh'][0])

CRIME_DIR  = BASE_DIR / "crime-data"
EXTRA_DIR  = BASE_DIR / "extra-data"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# enrichment file paths 
POPULATION_FILE = EXTRA_DIR / "population-23-25.csv"
IMD_FILE        = EXTRA_DIR / "Index_of_Multiple_Deprivation.xlsx"
SEVERITY_FILE   = EXTRA_DIR / "severity-score.xls"
OUTPUT_FILE     = OUTPUT_DIR / "crime_reporting_dataset.csv"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"CRIME_DIR  : {CRIME_DIR}")
print(f"EXTRA_DIR  : {EXTRA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# sanity check - fails if the folder structure is wrong
assert CRIME_DIR.exists(),  f"crime-data folder not found at: {CRIME_DIR}"
assert EXTRA_DIR.exists(),  f"extra-data folder not found at: {EXTRA_DIR}"
print("folder structure verified!")

BASE_DIR   : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning
CRIME_DIR  : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\crime-data
EXTRA_DIR  : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\extra-data
OUTPUT_DIR : c:\Users\mtbmo\OneDrive\Área de Trabalho\rockborne\projects\project6\data-cleaning\outputs
folder structure verified!


In [6]:
# force name configuration 

# canonical names used throughout the pipeline.
FORCES = [
    "Metropolitan Police Service",
    "West Midlands Police",
    "West Yorkshire Police",
    "Devon & Cornwall Police",
]

# maps filename keywords - canonical force name.
FORCE_FILE_KEYWORDS = {
    "metropolitan":       "Metropolitan Police Service",
    "west-midlands":      "West Midlands Police",
    "west-yorkshire":     "West Yorkshire Police",
    "devon-and-cornwall": "Devon & Cornwall Police",
}

# raw "falls within" / "reported by" variants - canonical names.
FORCE_NAME_MAP = {
    "Metropolitan Police Service":     "Metropolitan Police Service",
    "West Midlands Police":            "West Midlands Police",
    "West Yorkshire Police":           "West Yorkshire Police",
    "Devon & Cornwall Constabulary":   "Devon & Cornwall Police",
    "Devon and Cornwall Constabulary": "Devon & Cornwall Police",
    "Devon and Cornwall Police":       "Devon & Cornwall Police",
    "Devon & Cornwall Police":         "Devon & Cornwall Police",
}

#  LAD code (2021) - PFA canonical name 
# FORCE_POPULATION (hardcoded dictionary) has been removed and replaced by a
# data-driven LAD - PFA aggregation in Section 4b. This mapping is the bridge:
# 2021 ONS LAD codes - canonical PFA name. Only the four PFAs in scope are
# included; all other LADs in the CSV are filtered out.
#
# Assumptions:
# - 2021 LAD codes used as reference — boundaries last substantially revised
# - 2011, treated as stable through 2023-2026.
# - 2025 PFA boundaries used as static reference (midpoint of analysis window).
LAD_TO_PFA = {
    # Metropolitan Police Service (33 London Boroughs)
    "E09000001": "Metropolitan Police Service",  # City of London
    "E09000002": "Metropolitan Police Service",  # Barking and Dagenham
    "E09000003": "Metropolitan Police Service",  # Barnet
    "E09000004": "Metropolitan Police Service",  # Bexley
    "E09000005": "Metropolitan Police Service",  # Brent
    "E09000006": "Metropolitan Police Service",  # Bromley
    "E09000007": "Metropolitan Police Service",  # Camden
    "E09000008": "Metropolitan Police Service",  # Croydon
    "E09000009": "Metropolitan Police Service",  # Ealing
    "E09000010": "Metropolitan Police Service",  # Enfield
    "E09000011": "Metropolitan Police Service",  # Greenwich
    "E09000012": "Metropolitan Police Service",  # Hackney
    "E09000013": "Metropolitan Police Service",  # Hammersmith and Fulham
    "E09000014": "Metropolitan Police Service",  # Haringey
    "E09000015": "Metropolitan Police Service",  # Harrow
    "E09000016": "Metropolitan Police Service",  # Havering
    "E09000017": "Metropolitan Police Service",  # Hillingdon
    "E09000018": "Metropolitan Police Service",  # Hounslow
    "E09000019": "Metropolitan Police Service",  # Islington
    "E09000020": "Metropolitan Police Service",  # Kensington and Chelsea
    "E09000021": "Metropolitan Police Service",  # Kingston upon Thames
    "E09000022": "Metropolitan Police Service",  # Lambeth
    "E09000023": "Metropolitan Police Service",  # Lewisham
    "E09000024": "Metropolitan Police Service",  # Merton
    "E09000025": "Metropolitan Police Service",  # Newham
    "E09000026": "Metropolitan Police Service",  # Redbridge
    "E09000027": "Metropolitan Police Service",  # Richmond upon Thames
    "E09000028": "Metropolitan Police Service",  # Southwark
    "E09000029": "Metropolitan Police Service",  # Sutton
    "E09000030": "Metropolitan Police Service",  # Tower Hamlets
    "E09000031": "Metropolitan Police Service",  # Waltham Forest
    "E09000032": "Metropolitan Police Service",  # Wandsworth
    "E09000033": "Metropolitan Police Service",  # Westminster

    # West Midlands Police (7 metropolitan districts)
    "E08000025": "West Midlands Police",  # Birmingham
    "E08000026": "West Midlands Police",  # Coventry
    "E08000027": "West Midlands Police",  # Dudley
    "E08000028": "West Midlands Police",  # Sandwell
    "E08000029": "West Midlands Police",  # Solihull
    "E08000030": "West Midlands Police",  # Walsall
    "E08000031": "West Midlands Police",  # Wolverhampton

    # West Yorkshire Police (5 metropolitan districts)
    "E08000032": "West Yorkshire Police",  # Bradford
    "E08000033": "West Yorkshire Police",  # Calderdale
    "E08000034": "West Yorkshire Police",  # Kirklees
    "E08000035": "West Yorkshire Police",  # Leeds
    "E08000036": "West Yorkshire Police",  # Wakefield

    # Devon & Cornwall Police (Cornwall + Devon districts + Plymouth + Torbay)
    "E06000052": "Devon & Cornwall Police",  # Cornwall
    "E06000053": "Devon & Cornwall Police",  # Isles of Scilly
    "E10000008": "Devon & Cornwall Police",  # Devon (county)
    "E07000040": "Devon & Cornwall Police",  # East Devon
    "E07000041": "Devon & Cornwall Police",  # Exeter
    "E07000042": "Devon & Cornwall Police",  # Mid Devon
    "E07000043": "Devon & Cornwall Police",  # North Devon
    "E07000044": "Devon & Cornwall Police",  # South Hams
    "E07000045": "Devon & Cornwall Police",  # Teignbridge
    "E07000046": "Devon & Cornwall Police",  # Torridge
    "E07000047": "Devon & Cornwall Police",  # West Devon
    "E06000026": "Devon & Cornwall Police",  # Plymouth
    "E06000027": "Devon & Cornwall Police",  # Torbay
}

# ai disclosure - LAD code to PFA canonical name was derived with help from ai 

# severity score name mapping
SEVERITY_NAME_MAP = {
    "Metropolitan Police": "Metropolitan Police Service",
    "West Midlands":       "West Midlands Police",
    "West Yorkshire":      "West Yorkshire Police",
    "Devon and Cornwall":  "Devon & Cornwall Police",
}

# time period
START_MONTH = "2023-04"
END_MONTH   = "2026-03"
YEARS       = [2023, 2024, 2025, 2026]

print("Configuration loaded.")
print(f"Forces: {FORCES}")
print(f"LAD_TO_PFA: {len(LAD_TO_PFA)} LAD codes mapped across {len(FORCES)} PFAs")
print(f"Time window: {START_MONTH} → {END_MONTH}")


Configuration loaded.
Forces: ['Metropolitan Police Service', 'West Midlands Police', 'West Yorkshire Police', 'Devon & Cornwall Police']
LAD_TO_PFA: 58 LAD codes mapped across 4 PFAs
Time window: 2023-04 → 2026-03


---
## Section 1 - Ingestion Layer

Before loading the files, the folder structure will first be verified to ensure the expected monthly subdirectories are present. Reads only the four required force files for each month before concatenating them into a single raw DataFrame at the end.

Loading the entire dataset in a single operation would likely exceed the memory capacity of a standard machine, as the raw CSV files total approximately 1.3 GB across 36 months and four police forces. Therefore, processing the data in monthly batches allows progress to be monitored more effectively.

In [7]:
# folder structure validation
import re

monthly_folders = sorted([
    f.name for f in CRIME_DIR.iterdir()
    if f.is_dir() and re.match(r"\d{4}-\d{2}", f.name)
])

print(f"monthly folders found: {len(monthly_folders)}")
print(f"first: {monthly_folders[0]}  |  last: {monthly_folders[-1]}")

# check that we have the months wanted
expected_months = pd.period_range(start=START_MONTH, end=END_MONTH, freq="M").astype(str).tolist()
missing = set(expected_months) - set(monthly_folders)
extra   = set(monthly_folders) - set(expected_months)

if missing:
    print(f"missing folders: {sorted(missing)}")
else:
    print("all 36 expected monthly folders are present!!")

if extra:
    print(f"extra folders found (will be skipped): {sorted(extra)}")

monthly folders found: 36
first: 2023-04  |  last: 2026-03
all 36 expected monthly folders are present!!


In [8]:
# loop batch ingestion

chunks = []  # accumulate DataFrames here; concatenate once at the end

for month_folder in monthly_folders:
    folder_path = CRIME_DIR / month_folder
    files_in_folder = [f.name for f in folder_path.iterdir() if f.is_file()]
    month_chunks = []

    for keyword, force_name in FORCE_FILE_KEYWORDS.items():
        # Find the file whose name contains this force's keyword
        matches = [f for f in files_in_folder if keyword in f and f.endswith(".csv")]

        if not matches:
            print(f"  [{month_folder}] no file found for keyword '{keyword}'")
            continue

        filepath = folder_path / matches[0]

        # reads the CSV — dtype=str keeps everything as strings on load;
        # cast types in the cleaning stage
        df_chunk = pd.read_csv(filepath, dtype=str, low_memory=False)
        month_chunks.append(df_chunk)

    if month_chunks:
        month_df = pd.concat(month_chunks, ignore_index=True)
        chunks.append(month_df)
        print(f"  [{month_folder}] loaded {len(month_df):>8,} rows across {len(month_chunks)} force files")

print(f"all monthly batches processed. Concatenating into raw DataFrame...")
df_raw = pd.concat(chunks, ignore_index=True)
print(f"done!!!")

  [2023-04] loaded  154,835 rows across 4 force files
  [2023-05] loaded  164,785 rows across 4 force files
  [2023-06] loaded  174,208 rows across 4 force files
  [2023-07] loaded  166,604 rows across 4 force files
  [2023-08] loaded  160,283 rows across 4 force files
  [2023-09] loaded  160,805 rows across 4 force files
  [2023-10] loaded  168,389 rows across 4 force files
  [2023-11] loaded  157,010 rows across 4 force files
  [2023-12] loaded  155,906 rows across 4 force files
  [2024-01] loaded  153,592 rows across 4 force files
  [2024-02] loaded  150,436 rows across 4 force files
  [2024-03] loaded  158,105 rows across 4 force files
  [2024-04] loaded  156,655 rows across 4 force files
  [2024-05] loaded  170,061 rows across 4 force files
  [2024-06] loaded  167,186 rows across 4 force files
  [2024-07] loaded  174,052 rows across 4 force files
  [2024-08] loaded  168,178 rows across 4 force files
  [2024-09] loaded  160,218 rows across 4 force files
  [2024-10] loaded  169,155 